# Reflecting Telescope

A reflecting telescope, in its basic form, is a very simple design. Two surfaces both reflect light to focus it at a single point.

![Newton reflecting Telescope](./newton_reflecting_telescope.jpeg)

*Image from Wikipedia*

For this example, our telescope will be made of two convave mirrors. To spice things up, we'll say that the primary mirror is parabolic, and the secondary is spherical. Of course this can easily be changed, so feel free to download this notebook and play with it. In this example, we will keep the position of the two mirrors constant, and try to optimize the two mirrors curvatures jointly.

In [ ]:
import torchlensmaker as tlm

primary = tlm.Parabola(35., A=-0.0001, trainable=True)
secondary = tlm.SphereByCurvature(35., C=1/450.0, trainable=True)

source = tlm.Sequential(
    tlm.Gap(-100),
    tlm.PointSourceAtInfinity(beam_diameter=30),
)

model = tlm.Sequential(
    tlm.Gap(100),
    
    tlm.ReflectiveSurface(primary),
    tlm.Gap(-80),

    tlm.ReflectiveSurface(secondary),

    tlm.Gap(100),
)

target = tlm.FocalPoint()

optics = tlm.Sequential(source, model, target)

tlm.show2d(optics)
tlm.show3d(optics)

Now, as you can see light isn't being focused at all. We have wrapped both surfaces arguments in `tlm.parameter()`. Internally, this creates a `nn.Parameter()` so that PyTorch can optimize them. Let's run a standard Adam optimizer for 100 iterations, with 10 rays samples.

In [ ]:
# Optimization in 3D

import torch.optim as optim

optics.set_sampling3d(pupil=60)

root = tlm.SequentialData.empty(dim=3)
inputs = source.sequential(root)

tlm.optimize(
    model,
    inputs,
    target,
    optimizer = optim.Adam(optics.parameters(), lr=3e-5),
    num_iter = 150
).plot()

In [ ]:
# Optimization in 2D

import torch.optim as optim

optics.set_sampling2d(pupil=10)

root = tlm.SequentialData.empty(dim=2)
inputs = source.sequential(root)

tlm.optimize(
    model,
    inputs,
    target,
    optimizer = optim.Adam(optics.parameters(), lr=3e-6),
    num_iter = 150
).plot()

In [ ]:
tlm.show(optics, dim=2)
tlm.show(optics, dim=3)